# MATH4050 · Kickstarter model comparison

Gechleng Lim · **setup only; model experiments pending**

Foundation: [DATA3125 commit 21a6242](https://github.com/Crient/data3125-kickstarter-analysis/tree/21a6242b3ec3a5c78f3d9d3de5af00189edc7c1a).
Inherited notebooks/results remain DATA3125 references. Only the two data-setup
cells below execute. Every model section is a learning prompt for later sessions.

For each model: explain it, write/run a small block with ChatGPT, inspect outputs
and errors, then vary complexity and discuss training/validation behavior.
Leave automated cleanup/finalization until the experiments are understood.
Select this project's `.venv` kernel before running cells.

## Problem and founder launch-planning objective

- What can historical campaigns tell a founder about a proposed launch plan?
- Define reaching the funding goal as success (1), and failure as 0.
- Explain how a benchmark can inform planning while predictions remain limited.
- Which costs of missing failures should influence the metrics we prioritize?

Use exactly five predictors: USD funding goal, duration, launch month, main
category and country. Describe associations without causal or deployment claims.

## Dataset and evaluation design

Load the committed 63,075-campaign cohort without rebuilding it. Train on 2024
(20,581), tune/select on 2025 (25,706), then freeze models before refitting on
2024–2025 and comparing on 2026 (16,788). **2026 is a reused evaluation benchmark**;
its results have already been inspected in DATA3125. Do not tune/select on it.

Fit every learned transformation on training data only, including scaling,
encoding and PCA. If using cross-validation within 2024–2025, fit preprocessing
inside each fold. ID, timestamps, launch year, backers, final pledges and outcome
fields cannot be predictors. Launch year is for splitting; success is the target.

This is a **launch-year split**, not a January 1 knowledge simulation. Outcomes
can cross years; all snapshots were collected in 2026. Goal/category availability
at launch is unverified. Deduplication retains the first observation in sorted
CSV-part order per campaign/snapshot, then the latest snapshot per campaign;
that rule cannot establish which conflicting value is correct.

Eligibility requires successful/failed campaigns, deadlines before
**2026-10-02 00:00 UTC**, and state-change timestamps at/after launch and before
the same cutoff (through the end of October 1, 2026 UTC). Uneven snapshot coverage,
exclusion of unfinished/other-state campaigns, and cutoff selection limit
representativeness. See `docs/DATA3125_README.md` and `data/cohort_audit.json`.

**Setup check 1:** locate the project. Importing the inherited loader does not fit models.

In [1]:
from pathlib import Path
import hashlib
import sys

# Support opening from the repository root or from code/.
working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (working_directory, *working_directory.parents)
     if (candidate / "data/kickstarter_cleaned_2024_2026.csv").is_file()
     and (candidate / "code/kickstarter.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the project root or code/.")
code_directory = str(PROJECT_ROOT / "code")
if code_directory not in sys.path:
    sys.path.insert(0, code_directory)

from kickstarter import FEATURES, load_cleaned

print(f"Project: {PROJECT_ROOT.name}")
print(f"Python: {sys.version.split()[0]}")

Project: kickstarter-math4050-final-project
Python: 3.13.0


**Setup check 2:** verify the original CSV hash, schema, IDs, predictor values,
timestamps and exact year counts. These are data integrity checks; no models
are fitted, no model scores are computed, and no derived artifacts are written.

In [2]:
csv_path = PROJECT_ROOT / "data/kickstarter_cleaned_2024_2026.csv"
expected_sha256 = "b6c852abfce2d0dced78816e348edc59b3b919485f0401e123d7764a77fb3e2e"
if hashlib.sha256(csv_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError("The CSV differs from the verified DATA3125 source.")

campaigns = load_cleaned(csv_path)
expected_counts = {2024: 20581, 2025: 25706, 2026: 16788}
year_counts = campaigns.groupby("launch_year").size().sort_index()
expected_features = ["funding_goal_usd", "duration_days", "launch_month", "main_category", "country"]
if len(campaigns) != 63075 or campaigns["id"].nunique() != 63075:
    raise ValueError("Expected 63,075 unique campaigns.")
if year_counts.to_dict() != expected_counts:
    raise ValueError(f"Unexpected launch-year counts: {year_counts.to_dict()}")
if FEATURES != expected_features:
    raise ValueError("The five allowed predictors have changed.")

print("Verified original CSV: 63,075 unique campaigns; schema and feature checks passed.")
print("Predictors:", ", ".join(FEATURES))
print("Target: success (failure=0, success=1). Model experiments are pending.")
year_counts.rename("campaigns").to_frame()

Verified original CSV: 63,075 unique campaigns; schema and feature checks passed.
Predictors: funding_goal_usd, duration_days, launch_month, main_category, country
Target: success (failure=0, success=1). Model experiments are pending.


,campaigns
launch_year,
2024,20581
2025,25706
2026,16788


## Majority-class baseline

**Pending · learning session**

- Determine the majority class using training labels only. Why is it a necessary comparison?
- Before coding, predict what happens to failure recall when every prediction is success.
- Plan accuracy and failure-class (0) precision, recall and F1. Explain undefined precision and any `zero_division=0` convention.
- Write/run the baseline together in a later session; do not compute it during setup.

## Logistic Regression

**Pending · learning session**

- Explain log-odds, regularization and why numeric feature scaling matters.
- Build a training-only preprocessing pipeline together. How should unknown categories/countries be handled?
- Explore regularization strength using 2024 training and 2025 validation, recording the train/validation gap.
- Compare failure metrics with the baseline and discuss what this model can miss. Existing DATA3125 results are references only.

## Decision Tree

**Pending · learning session**

- Explain splits, depth, leaf size and overfitting using the five predictors.
- Build the pipeline together and vary depth or minimum leaf size on training/validation only.
- Plot complexity versus training and validation performance; explain underfit/overfit regions.
- Discuss failure errors and unknown categories. Keep the inherited fitted-tree demo unchanged.

## Random Forest

**Pending · learning session**

- Explain bagging, feature subsampling and how a forest differs from one tree.
- Build and run a small forest together before considering a bounded parameter experiment.
- Examine depth/leaf size, tree count and training/validation gaps without consulting 2026 performance.
- Discuss computation, reproducibility and limits of feature-importance interpretations.

## KNN

**Pending · learning session**

- Explain distance, neighbor count and why scaling changes the model.
- Discuss mixed numeric and one-hot categorical inputs before building the pipeline.
- Vary k on 2024/2025; inspect training/validation gaps and failure recall.
- Discuss dimensionality, prediction cost and any training-only sampling needed for an affordable experiment.

## SVM

**Pending · learning session**

- Explain margins, C and the difference between linear and kernel decision boundaries.
- Plan scaled/encoded inputs, training-only preprocessing and an affordable experiment.
- Explore a small, justified C/kernel parameter range using 2025 validation only.
- Discuss runtime and the meaning of decision scores before interpreting errors.

## PCA with a downstream classifier

**Pending · learning session**

- Explain variance, principal components and why PCA itself is not a classifier.
- Decide which representation to reduce and justify scaling, encoding and sparse/dense memory handling.
- Fit PCA only within each training pipeline/fold, then tune component count and classifier settings on 2024–2025.
- Compare with the same classifier without PCA; discuss validation performance, variance retained and interpretability. Do not fit PCA during setup.

## Training versus validation and bias–variance analysis

**Pending · learning session**

- Collect comparable training and 2025 validation metrics for the completed experiments.
- Plot performance against justified complexity settings and interpret the gaps.
- Distinguish model complexity from sample size and explain what the curves support.
- Fix the selection metric and parameters before refitting on 2024–2025. Keep 2026 out of these choices.

## Final model comparison and error analysis

**Pending · learning session**

- After the learning experiments are complete, freeze the models and refit using 2024–2025 only.
- Compare on the reused 2026 benchmark and label it explicitly; it is not an untouched test set.
- Report accuracy, failure precision/recall/F1 and confusion matrices (actual rows/predicted columns; label order [0, 1]).
- Examine false successes (actual failure 0, predicted success 1), subgroup sample sizes and unknown category/country behavior.
- Separate new MATH4050 results from inherited DATA3125 references. Do not use benchmark errors for further tuning.

## Conclusions and limitations

**Pending · learning session**

- Answer the founder planning question using the completed comparisons and explain missed failures.
- Discuss sampling, end-of-October-1 cutoff selection, duplicate conflicts and exclusions from the binary target.
- Explain calendar-crossing outcomes, retrospective feature availability and prior exposure to the 2026 benchmark.
- State unknown-category handling, uncertainty for small groups and computational/interpretability tradeoffs.
- Describe associations without claiming causation or deployment performance. Word/PDF writing is a later phase; keep reports/ ready.